# Full Initialization Workflow - Single `.mo` File

This notebook initializes one dynamic Modelica model from a single `.mo` file.

It takes the input model from `models/`, creates the auxiliary model, uses it to compute the initialization values, and generates the initialized model in `outputs/`.


In [ ]:
include("../scripts/dictionaries.jl")
include("../scripts/helpers.jl")

using .WorkflowHelpers
using OMJulia

In [ ]:
# User configuration: edit the values below for your model.

# Name of the dynamic model to initialize.
# The corresponding .mo file must be available in the models folder.
MODEL = "DIGrid"

# Your Dynawo installation (used only for its Modelica Standard Library).
DYNAWO_DIR = "/home/clarafercas/dynawo"
MODELICA_PKG_PATH = "$DYNAWO_DIR/OpenModelica/lib/omlibrary/Modelica/package.mo"

# Dynawo Modelica library from this repo.
DYNAWO_PKG_PATH = abspath("../dynawo_library/Dynawo/package.mo")

# Optional settings

# INIT model selection for components with multiple INIT profiles
INIT_MODEL_BY_COMPONENT = Dict{String, String}(
    # "generatorSynchronous" => "GeneratorSynchronousInt_INIT",
)

# Leave empty to disable slack-specific handling.
SLACK_COMPONENT = "inertialGrid1";


In [ ]:
# Internal paths derived from the user configuration above.
MODEL_DIR = abspath("models")
OUTPUT_DIR = abspath("outputs")

MODEL_FILE_PATH = joinpath(MODEL_DIR, MODEL * ".mo")

AUX_MODEL = MODEL * "_auxiliary"
AUX_FILE_PATH = joinpath(OUTPUT_DIR, AUX_MODEL * ".mo")

INITIALIZED_MODEL = MODEL * "_initialized"
INITIALIZED_FILE_PATH = joinpath(OUTPUT_DIR, INITIALIZED_MODEL * ".mo")

isfile(MODEL_FILE_PATH) || error("Model file not found: $MODEL_FILE_PATH")
isfile(DYNAWO_PKG_PATH) || error("Dynawo package not found: $DYNAWO_PKG_PATH")
isfile(MODELICA_PKG_PATH) || error("Modelica package not found: $MODELICA_PKG_PATH")
isdir(OUTPUT_DIR) || error("Output directory not found: $OUTPUT_DIR")

## Build Auxiliary Model

The original dynamic model is loaded and checked before any changes are made. An auxiliary model is then generated and saved for the initialization simulation.


In [ ]:
DynamicOMC = OMJulia.OMCSession()
omc_call(DynamicOMC, "loadFile(\"$MODELICA_PKG_PATH\")")
omc_call(DynamicOMC, "loadModel(Complex)")
omc_call(DynamicOMC, "loadModel(ModelicaServices)")
omc_call(DynamicOMC, "loadFile(\"$DYNAWO_PKG_PATH\")")
omc_call(DynamicOMC, "loadFile(\"$MODEL_FILE_PATH\")")

check_user_configuration_single(DynamicOMC;
    model = MODEL,
    slack_component = SLACK_COMPONENT,
    init_model_by_component = INIT_MODEL_BY_COMPONENT,
)

In [ ]:
source_components = get_all_components(DynamicOMC, MODEL)

# Drop any class left over from a previous run. A missing class is fine here.
sendExpression(DynamicOMC, "deleteClass($AUX_MODEL)")
println("Creating auxiliary model $AUX_MODEL from $MODEL")
omc_call(DynamicOMC, "copyClass($MODEL, \"$AUX_MODEL\")")

apply_replacements!(
    DynamicOMC,
    MODEL,
    AUX_MODEL,
    source_components,
    SLACK_COMPONENT,
)

delete_connections!(DynamicOMC, AUX_MODEL, source_components)
delete_components!(DynamicOMC, AUX_MODEL, source_components)

add_init_models!(
    DynamicOMC,
    MODEL,
    AUX_MODEL,
    source_components,
    INIT_MODEL_BY_COMPONENT,
    SLACK_COMPONENT,
)

apply_LF_modifiers!(
    DynamicOMC,
    MODEL,
    AUX_MODEL,
    source_components,
)

add_init_equations!(
    DynamicOMC,
    MODEL,
    AUX_MODEL,
    source_components,
    INIT_MODEL_BY_COMPONENT,
    SLACK_COMPONENT,
)

clean_aux_equations!(DynamicOMC, AUX_MODEL, source_components, SLACK_COMPONENT)
omc_call(DynamicOMC, "saveModel(\"$AUX_FILE_PATH\", $AUX_MODEL)")

println("Wrote auxiliary model: ", AUX_FILE_PATH)


## Extract Initialization Values

The written auxiliary model is loaded independently, checked, and simulated. Its simulation outputs provide the initialization values to apply to the original dynamic model.


In [ ]:
AuxiliaryOMC = OMJulia.OMCSession()
omc_call(AuxiliaryOMC, "loadModel(Complex)")
omc_call(AuxiliaryOMC, "loadModel(ModelicaServices)")
omc_call(AuxiliaryOMC, "loadFile(\"$MODELICA_PKG_PATH\")")
omc_call(AuxiliaryOMC, "loadFile(\"$DYNAWO_PKG_PATH\")")
omc_call(AuxiliaryOMC, "loadFile(\"$AUX_FILE_PATH\")")

omc_call(AuxiliaryOMC, "checkModel($AUX_MODEL)", parsed = false)
println("Auxiliary model checked successfully.")


In [ ]:
ModelicaSystem(AuxiliaryOMC, AUX_FILE_PATH, AUX_MODEL, [MODELICA_PKG_PATH, DYNAWO_PKG_PATH])
simulate(AuxiliaryOMC, resultfile = AUX_MODEL * "_res.mat")

initializable_components = get_initializable_components(source_components, INIT_MODEL_BY_COMPONENT)

init_values_by_component = extract_all_initialization_values(
    AuxiliaryOMC,
    source_components,
    INIT_MODEL_BY_COMPONENT,
)

initialized_component_names = sort(collect(keys(init_values_by_component)))
println("Extracted initialization values for: ", join(initialized_component_names, ", "))


## Build Initialized Model

A copy of the validated original dynamic model is populated with the values extracted from the auxiliary simulation and saved as the initialized model.


In [ ]:
# Drop any class left over from a previous run. A missing class is fine here.
sendExpression(DynamicOMC, "deleteClass($INITIALIZED_MODEL)")
println("Creating initialized model $INITIALIZED_MODEL from $MODEL")
omc_call(DynamicOMC, "copyClass($MODEL, \"$INITIALIZED_MODEL\")")

apply_initialization_modifiers!(
    DynamicOMC,
    INITIALIZED_MODEL,
    initializable_components,
    init_values_by_component,
    INIT_MODEL_BY_COMPONENT,
)

omc_call(DynamicOMC, "saveModel(\"$INITIALIZED_FILE_PATH\", $INITIALIZED_MODEL)")
println("Wrote initialized model: ", INITIALIZED_FILE_PATH)


## Validate Initialized Model

The generated initialized model is loaded independently and simulated to verify that the final output can run.


In [ ]:
InitializedOMC = OMJulia.OMCSession()
omc_call(InitializedOMC, "loadModel(Complex)")
omc_call(InitializedOMC, "loadModel(ModelicaServices)")
ModelicaSystem(InitializedOMC, INITIALIZED_FILE_PATH, INITIALIZED_MODEL, [MODELICA_PKG_PATH, DYNAWO_PKG_PATH])

omc_call(InitializedOMC, "checkModel($INITIALIZED_MODEL)", parsed = false)
println("Initialized model checked successfully.")

initialized_resultfile_prefix = INITIALIZED_MODEL
simflags = simulation_flags_without_log_stats(InitializedOMC, INITIALIZED_MODEL)

sendExpression(
    InitializedOMC,
    "simulate($INITIALIZED_MODEL, outputFormat=\"csv\", fileNamePrefix=\"$initialized_resultfile_prefix\", simflags=\"$simflags\")",
    parsed = false,
)

initialized_resultfile = joinpath(
    getWorkDirectory(InitializedOMC),
    initialized_resultfile_prefix * "_res.csv",
)
println("Initialized model simulated successfully.")


## Optional: Plot Simulation Result

The final cell is an optional example of visualizing a variable from the initialized-model simulation result.


In [ ]:
using Plots, DataFrames, CSV

# Select a variable to visualize from the initialized simulation.
PLOT_VARIABLE = "deltaFrequency"

initialized_df = DataFrame(CSV.File(initialized_resultfile))
PLOT_VARIABLE in names(initialized_df) ||
    error("PLOT_VARIABLE \"$PLOT_VARIABLE\" is not a variable in the simulation result.")

plotlyjs()
p = plot(initialized_df[!, "time"], initialized_df[!, PLOT_VARIABLE], label = [PLOT_VARIABLE])
plot!(p, legend = :bottomright, titlefontsize = 12, labelfontsize = 10)
title!(p, "Initialized dynamic model response")
xlabel!(p, "Time (s)")
ylabel!(p, PLOT_VARIABLE)
